# 04 — Model Selection & 50-Split Robust Validation

**Phase A (Development, seed=42)**: For each target, Optuna TPE tunes 6 models
(top 6 from N01 imputer evaluation) on N03 SFFS-selected features. **Selection
criterion: lowest 5-fold CV RMSE** (consistent with N01-N03 using RMSE as
primary metric). Best model + hyperparams locked per target. Evaluate on holdout
set for single-split reference.

**Phase B (Evaluation, 50 splits)**: On full cleaned data (494 samples, NaN
targets preserved), 50 independent random 70/30 splits. Per-split pipeline:

split → MICE(ExtraTrees, fit on train) → StandardScaler(fit on train)
      → train(Phase A locked model + hyperparams) → evaluate(R², RMSE, MAE)

MICE imputation happens **after splitting** — fit on train only, transform both.
X has no NaN; MICE stacks selected features X + target y, using fully-observed
X to predict and fill missing y values. Same pattern as N01 §6.

Each target evaluated independently with its own locked model + features.

**Design choices locked from seed=42**: imputer type = ExtraTrees (N01),
features = N03 SFFS, model + hyperparams = Phase A.

**Input**: `metal_cleaned.csv`, `X.npy`, `metal_train_imputed.csv`,
`metal_holdout_imputed.csv`, `X_train.npy`, `X_test.npy`,
`selected_features.json`, `optimal_features.json`, `best_imputer.json`

**Output**: `best_models/`, `model_selection_results.json`,
`repeated_split_results.json`, `plot_phase_b.npz`

In [1]:
import os, json, pickle, time, warnings, numpy as np, pandas as pd
from collections import OrderedDict
warnings.filterwarnings('ignore')

RNG_SEED = 42; np.random.seed(RNG_SEED)
DATA_DIR = os.path.join('..', 'data')
MODEL_DIR = os.path.join('..', 'best_models')
os.makedirs(MODEL_DIR, exist_ok=True)

TARGET_COLS = ['YS', 'UTS', 'El']

# Mass fraction + process columns (raw features, from csv)
RAW_COLS = ['Si','Fe','Cu','Mn','Mg','Cr','Zn','V','Ti','Zr','Li','Ni','Be','Sc','Tsol','Tage','tage']

# ── N03 SFFS-selected features (per-target feature names) ──
with open(os.path.join(DATA_DIR, 'selected_features.json')) as f:
    selected_sets = json.load(f)

# ── Engineered feature names (330 names, from N02) ──
# Used only for mapping engineered feature names → X array column indices.
# Raw features (mass fractions, process params) come from csv, not X arrays.
with open(os.path.join(DATA_DIR, 'optimal_features.json')) as f:
    ALL_FEATURES = json.load(f)['feature_names_all']

# ── Helper: build feature matrix from selected feature names ──
# Raw features (mass fractions + process) from df; engineered features from X array.
# Same approach as N03 build_X().
def build_X(flist, df, X_arr):
    """Build feature matrix: raw cols from df, engineered cols from X_arr."""
    raw = [c for c in flist if c in RAW_COLS]
    eng = [c for c in flist if c not in RAW_COLS]
    parts = []
    if raw:
        parts.append(df[raw].values.astype(float))
    if eng:
        idx = [ALL_FEATURES.index(c) for c in eng if c in ALL_FEATURES]
        if idx:
            parts.append(X_arr[:, idx])
    if len(parts) == 0:
        raise ValueError(f'No features found for {flist}')
    return np.column_stack(parts) if len(parts) > 1 else parts[0]

# ── Phase A: seed=42 train/holdout (targets already imputed by N01 MICE) ──
df_train = pd.read_csv(os.path.join(DATA_DIR, 'metal_train_imputed.csv'))
df_holdout = pd.read_csv(os.path.join(DATA_DIR, 'metal_holdout_imputed.csv'))
X_train_arr = np.load(os.path.join(DATA_DIR, 'X_train.npy'))   # 345 × 330 engineered features
X_test_arr  = np.load(os.path.join(DATA_DIR, 'X_test.npy'))    # 149 × 330

X_dev = X_train_arr
y_dev = df_train[TARGET_COLS].values.astype(float)
X_holdout = X_test_arr
y_holdout = df_holdout[TARGET_COLS].values.astype(float)

print(f'Train set (seed=42): {len(df_train)} samples (targets imputed by N01 MICE)')
print(f'Holdout set:         {len(df_holdout)} samples (targets imputed by N01 MICE)')
assert df_train[TARGET_COLS].isna().sum().sum() == 0, 'Train targets should be fully imputed!'
assert df_holdout[TARGET_COLS].isna().sum().sum() == 0, 'Holdout targets should be fully imputed!'

# ── Phase B: full cleaned data (NaN targets preserved) ──
# Imputation happens PER SPLIT (fit MICE on train, transform test).
# This prevents information leakage from test into training imputation.
df_full = pd.read_csv(os.path.join(DATA_DIR, 'metal_cleaned.csv'))
X_full = np.load(os.path.join(DATA_DIR, 'X.npy'))              # 494 × 330 engineered features
y_full = df_full[TARGET_COLS].values.astype(float)              # NaN preserved

print(f'\nFull dataset: {len(df_full)} samples x {X_full.shape[1]} engineered features')
print(f'  NaN in y_full: {np.isnan(y_full).sum()} cells '
      f'(YS={df_full["YS"].isna().sum()}, UTS={df_full["UTS"].isna().sum()}, '
      f'El={df_full["El"].isna().sum()}) — filled by per-split MICE in Phase B')

print(f'\nN03 SFFS-selected features (per target):')
for t in TARGET_COLS:
    feats = selected_sets[t]
    n_raw = len([f for f in feats if f in RAW_COLS])
    n_eng = len(feats) - n_raw
    print(f'  {t}: {len(feats)} features ({n_raw} raw + {n_eng} engineered)')
    print(f'       {", ".join(feats)}')

Train set (seed=42): 345 samples (targets imputed by N01 MICE)
Holdout set:         149 samples (targets imputed by N01 MICE)

Full dataset: 494 samples x 330 engineered features
  NaN in y_full: 101 cells (YS=49, UTS=15, El=37) — filled by per-split MICE in Phase B

N03 SFFS-selected features (per target):
  YS: 9 features (0 raw + 9 engineered)
       B_cxp_Cu_x_LMP, B_wrt_total_solute, C_prc_Tage_log_tage, B_cxp_Zr_x_Tage, C_prc_Tsol_Tage, B_cxp_Mg_x_Tsol, B_cxp_Fe_x_Tsol, B_cxp_Li_x_log_tage, B_cxp_Cr_x_Tage
  UTS: 5 features (0 raw + 5 engineered)
       D_thm_delta, C_prc_larson_miller, D_el_specific_E, B_wrt_solidus_depression, C_prc_Tsol_Tage
  El: 7 features (4 raw + 3 engineered)
       D_delta_Tm_abs_to_Al, Tage, tage, Tsol, D_delta_IE1_abs_to_Al, D_delta_chi_to_Al, Zn


## 1. Model Lineup — Top 6 from N01 Imputer Evaluation

N01 evaluated 11 models as MICE imputer estimators (5-fold CV, wNRMSE).
Top 6, ranked by wNRMSE:

| Rank | Model | N01 wNRMSE |
|------|-------|------------|
| 1 | **ExtraTrees** | 0.270 |
| 2 | **GradientBoosting** | 0.297 |
| 3 | **LightGBM** | 0.308 |
| 4 | **LassoCV** | 0.336 |
| 5 | **RandomForest** | 0.341 |
| 6 | **KNeighbors** | 0.363 |

LassoCV has built-in CV for alpha selection (no Optuna needed).
Remaining 5 models use Optuna TPE for hyperparameter tuning.

In [2]:
from sklearn.linear_model import LassoCV, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              GradientBoostingRegressor)
from sklearn.model_selection import cross_validate, KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from lightgbm import LGBMRegressor

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ══════════════════════════════════════════════════════════════
# Optuna objective functions (search spaces)
# ══════════════════════════════════════════════════════════════

def _obj_et(trial):
    return ExtraTreesRegressor(
        n_estimators=trial.suggest_int('n_estimators', 100, 500, step=50),
        max_depth=trial.suggest_int('max_depth', 3, 20),
        min_samples_split=trial.suggest_int('min_samples_split', 2, 20),
        min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 10),
        random_state=RNG_SEED, n_jobs=-1)

def _obj_gbr(trial):
    return GradientBoostingRegressor(
        n_estimators=trial.suggest_int('n_estimators', 100, 800, step=50),
        max_depth=trial.suggest_int('max_depth', 3, 8),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        random_state=RNG_SEED)

def _obj_lgbm(trial):
    return LGBMRegressor(
        n_estimators=trial.suggest_int('n_estimators', 100, 800, step=50),
        num_leaves=trial.suggest_int('num_leaves', 8, 127),
        min_child_samples=trial.suggest_int('min_child_samples', 5, 30),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        reg_alpha=trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        random_state=RNG_SEED, verbose=-1, n_jobs=-1)

def _obj_rf(trial):
    return RandomForestRegressor(
        n_estimators=trial.suggest_int('n_estimators', 100, 500, step=50),
        max_depth=trial.suggest_int('max_depth', 3, 20),
        min_samples_split=trial.suggest_int('min_samples_split', 2, 20),
        min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 10),
        random_state=RNG_SEED, n_jobs=-1)

def _obj_knn(trial):
    return KNeighborsRegressor(
        n_neighbors=trial.suggest_int('n_neighbors', 2, 20),
        weights=trial.suggest_categorical('weights', ['uniform', 'distance']))

# ── Build model from tuned params (no Optuna trial needed) ──
# Used by Phase A holdout, Phase B, and model saving.
def _build_tuned_model(model_name, best_params):
    """Build a model instance from a best_params dict."""
    bp = best_params if best_params else {}
    if model_name == 'ExtraTrees':
        return ExtraTreesRegressor(
            n_estimators=bp.get('n_estimators', 200),
            max_depth=bp.get('max_depth', 10),
            min_samples_split=bp.get('min_samples_split', 5),
            min_samples_leaf=bp.get('min_samples_leaf', 3),
            random_state=RNG_SEED, n_jobs=-1)
    elif model_name == 'GradientBoosting':
        return GradientBoostingRegressor(
            n_estimators=bp.get('n_estimators', 200),
            max_depth=bp.get('max_depth', 5),
            learning_rate=bp.get('learning_rate', 0.1),
            subsample=bp.get('subsample', 0.8),
            random_state=RNG_SEED)
    elif model_name == 'LightGBM':
        return LGBMRegressor(
            n_estimators=bp.get('n_estimators', 200),
            num_leaves=bp.get('num_leaves', 31),
            min_child_samples=bp.get('min_child_samples', 10),
            learning_rate=bp.get('learning_rate', 0.1),
            subsample=bp.get('subsample', 0.8),
            colsample_bytree=bp.get('colsample_bytree', 0.8),
            reg_lambda=bp.get('reg_lambda', 1.0),
            reg_alpha=bp.get('reg_alpha', 0.1),
            random_state=RNG_SEED, verbose=-1, n_jobs=-1)
    elif model_name == 'LassoCV':
        # Phase A tuned alpha via LassoCV's built-in CV.  Use plain Lasso
        # with the locked alpha — NOT LassoCV which would re-run its own
        # CV and pick a different alpha, breaking the "locked" contract.
        return Lasso(alpha=bp.get('alpha', 1.0), max_iter=5000,
                     random_state=RNG_SEED)
    elif model_name == 'RandomForest':
        return RandomForestRegressor(
            n_estimators=bp.get('n_estimators', 200),
            max_depth=bp.get('max_depth', 10),
            min_samples_split=bp.get('min_samples_split', 5),
            min_samples_leaf=bp.get('min_samples_leaf', 3),
            random_state=RNG_SEED, n_jobs=-1)
    elif model_name == 'KNeighbors':
        return KNeighborsRegressor(
            n_neighbors=bp.get('n_neighbors', 5),
            weights=bp.get('weights', 'uniform'))
    else:
        raise ValueError(f'Unknown model: {model_name}')

# ── Model registry — top 6 from N01, ordered by N01 wNRMSE ──
# Format: {key: (display_name, objective_fn, n_trials)}
# LassoCV self-tunes alpha via built-in 5-fold CV (no Optuna needed)
MODEL_REGISTRY = OrderedDict([
    ('ExtraTrees',       ('ExtraTrees',        _obj_et,   50)),
    ('GradientBoosting', ('GradientBoosting',  _obj_gbr,  50)),
    ('LightGBM',         ('LightGBM',          _obj_lgbm, 50)),
    ('LassoCV',          ('LassoCV',           None,        0)),  # built-in CV
    ('RandomForest',     ('RandomForest',      _obj_rf,    50)),
    ('KNeighbors',       ('KNeighbors',        _obj_knn,   50)),
])

print(f'\nModel registry ({len(MODEL_REGISTRY)} models, top 6 from N01):')
for mname, (label, _, n_trials) in MODEL_REGISTRY.items():
    tune_info = f'{n_trials} Optuna trials' if n_trials > 0 else 'built-in 5-fold CV'
    print(f'  {label:20s} — {tune_info}')



Model registry (6 models, top 6 from N01):
  ExtraTrees           — 50 Optuna trials
  GradientBoosting     — 50 Optuna trials
  LightGBM             — 50 Optuna trials
  LassoCV              — built-in 5-fold CV
  RandomForest         — 50 Optuna trials
  KNeighbors           — 50 Optuna trials


## 2. Phase A — Model Selection + Optuna Tuning (seed=42 dev set)

**Goal**: For each target, select the best model + hyperparameters from the
N01 top 6 lineup. **Selection criterion: lowest 5-fold CV RMSE** — consistent
with N01-N03 using RMSE as the primary metric throughout the pipeline.

For each target, using N03 SFFS-selected features:

1. **Optuna TPE** tunes 5 models (ExtraTrees, GradientBoosting, LightGBM,
   RandomForest, KNeighbors). 50 trials each, 5-fold CV, scoring =
   `neg_root_mean_squared_error`. TPESampler(multivariate=True) with
   MedianPruner stops unpromising trials early.
2. **LassoCV** self-tunes alpha via built-in 5-fold CV (no Optuna needed).
3. **Best model per target** = lowest CV RMSE → locked for Phase B.
4. Train winner on full dev set → evaluate on holdout (single-split reference).

In [3]:
from sklearn.dummy import DummyRegressor

# ── Optuna objective (with pruning) ──
# Scoring = neg_root_mean_squared_error (same as N02/N03).
# Maximizing neg_RMSE = minimizing RMSE.
# R², MAE, and RMSE std are tracked via user_attrs — no post-hoc CV needed.
def _optuna_objective(trial, X, y, model_name):
    _, obj_fn, _ = MODEL_REGISTRY[model_name]
    model = obj_fn(trial)
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    kf = KFold(n_splits=5, shuffle=True, random_state=RNG_SEED)
    rmse_scores, r2_scores, mae_scores = [], [], []
    for step, (tr_idx, va_idx) in enumerate(kf.split(X)):
        pipe.fit(X[tr_idx], y[tr_idx])
        yp = pipe.predict(X[va_idx])
        rmse_scores.append(-np.sqrt(mean_squared_error(y[va_idx], yp)))
        r2_scores.append(r2_score(y[va_idx], yp))
        mae_scores.append(-mean_absolute_error(y[va_idx], yp))
        trial.report(np.mean(rmse_scores), step)
        if trial.should_prune():
            raise optuna.TrialPruned()
    rmse_arr = -np.array(rmse_scores)
    r2_arr   = np.array(r2_scores)
    mae_arr  = -np.array(mae_scores)
    trial.set_user_attr('cv_rmse_std', float(np.std(rmse_arr)))
    trial.set_user_attr('cv_r2',       float(np.mean(r2_arr)))
    trial.set_user_attr('cv_r2_std',   float(np.std(r2_arr)))
    trial.set_user_attr('cv_mae',      float(np.mean(mae_arr)))
    trial.set_user_attr('cv_mae_std',  float(np.std(mae_arr)))
    return np.mean(rmse_scores)  # maximize neg_RMSE = minimize RMSE

sampler = TPESampler(seed=RNG_SEED, multivariate=True, n_startup_trials=10)
pruner  = MedianPruner(n_startup_trials=10, n_warmup_steps=3)

phase_a_results = {}  # {target: {best_model, best_params, cv_rmse, all_cv_scores}}

for target_idx, target_name in enumerate(TARGET_COLS):
    sel_feat_names = selected_sets[target_name]
    # Build feature matrix: raw features from csv, engineered from X array
    X_sel = build_X(sel_feat_names, df_train, X_dev)
    y_t = y_dev[:, target_idx]

    print(f'\n{"="*80}')
    print(f'  PHASE A — {target_name} ({len(sel_feat_names)} N03 features, {len(y_t)} samples)')
    print(f'  Selection criterion: lowest CV RMSE (scoring=neg_root_mean_squared_error)')
    print(f'{"="*80}')

    # Baseline
    dummy = DummyRegressor(strategy='mean')
    cv_dummy = cross_validate(dummy, X_sel, y_t, cv=5,
                              scoring=['r2', 'neg_root_mean_squared_error'])
    r2_dummy = np.mean(cv_dummy['test_r2'])
    rmse_dummy = float(np.mean(-cv_dummy['test_neg_root_mean_squared_error']))
    print(f'  {"Model":20s} {"RMSE (CV)":>24s} {"R²":>22s} {"MAE":>22s} {"Time":>8s}')
    print(f'  {"─"*20} {"─"*24} {"─"*22} {"─"*22} {"─"*8}')
    print(f'  {"Dummy(mean)":20s} {rmse_dummy:>8.2f} ± {np.std(-cv_dummy["test_neg_root_mean_squared_error"]):.1f}  '
          f'{r2_dummy:>8.4f}')

    target_cv = {}

    for mname, (label, obj_fn, n_trials) in MODEL_REGISTRY.items():
        t0 = time.perf_counter()
        if mname == 'LassoCV':
            # LassoCV self-tunes alpha via built-in CV
            pipe = Pipeline([('scaler', StandardScaler()),
                            ('model', LassoCV(alphas=np.logspace(-4, 2, 50),
                                              cv=5, random_state=RNG_SEED, max_iter=5000))])
            cv = cross_validate(pipe, X_sel, y_t, cv=5,
                               scoring=['r2', 'neg_root_mean_squared_error',
                                        'neg_mean_absolute_error'])
            best_cv_rmse = float(np.mean(-cv['test_neg_root_mean_squared_error']))
            best_cv_r2   = float(np.mean(cv['test_r2']))
            best_cv_mae  = float(np.mean(-cv['test_neg_mean_absolute_error']))
            best_cv_rmse_std = float(np.std(-cv['test_neg_root_mean_squared_error']))
            best_cv_r2_std   = float(np.std(cv['test_r2']))
            best_cv_mae_std  = float(np.std(-cv['test_neg_mean_absolute_error']))
            # cross_validate fits clones — need to fit original to extract alpha_
            pipe.fit(X_sel, y_t)
            best_params = {'alpha': float(pipe.named_steps['model'].alpha_)}
            elapsed = time.perf_counter() - t0
            print(f'  {label:20s} {best_cv_rmse:>8.2f} ± {best_cv_rmse_std:<5.1f}  '
                  f'{best_cv_r2:>8.4f} ± {best_cv_r2_std:<7.4f}  '
                  f'{best_cv_mae:>8.2f} ± {best_cv_mae_std:<5.1f}  {elapsed:>6.1f}s')
        else:
            # Optuna TPE tuning (maximize neg_RMSE; all metrics from best_trial.user_attrs)
            study = optuna.create_study(
                direction='maximize', sampler=sampler, pruner=pruner,
                study_name=f'{target_name}_{mname}')
            study.optimize(
                lambda trial, mn=mname: _optuna_objective(trial, X_sel, y_t, mn),
                n_trials=n_trials, show_progress_bar=True, n_jobs=1)
            best_params = study.best_params
            best_cv_rmse     = float(-study.best_value)
            best_cv_rmse_std = float(study.best_trial.user_attrs['cv_rmse_std'])
            best_cv_r2       = float(study.best_trial.user_attrs['cv_r2'])
            best_cv_r2_std   = float(study.best_trial.user_attrs['cv_r2_std'])
            best_cv_mae      = float(study.best_trial.user_attrs['cv_mae'])
            best_cv_mae_std  = float(study.best_trial.user_attrs['cv_mae_std'])
            elapsed = time.perf_counter() - t0
            n_done = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
            n_pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
            print(f'  {label:20s} {best_cv_rmse:>8.2f} ± {best_cv_rmse_std:<5.1f}  '
                  f'{best_cv_r2:>8.4f} ± {best_cv_r2_std:<7.4f}  '
                  f'{best_cv_mae:>8.2f} ± {best_cv_mae_std:<5.1f}  {elapsed:>6.1f}s  '
                  f'trials: {n_done} done/{n_pruned} pruned')
            print(f'  {"":20s} best params: {best_params}')

        target_cv[mname] = {
            'best_params': best_params,
            'cv_rmse': best_cv_rmse,
            'cv_rmse_std': best_cv_rmse_std,
            'cv_r2': best_cv_r2,
            'cv_r2_std': best_cv_r2_std,
            'cv_mae': best_cv_mae,
            'cv_mae_std': best_cv_mae_std,
            'train_time_s': elapsed,
        }

    # Select best model = lowest CV RMSE
    best_name = min(target_cv, key=lambda k: target_cv[k]['cv_rmse'])
    print(f'  >>> Best: {best_name} (CV RMSE={target_cv[best_name]["cv_rmse"]:.2f})')

    phase_a_results[target_name] = {
        'best_model': best_name,
        'best_params': target_cv[best_name]['best_params'],
        'cv_rmse': target_cv[best_name]['cv_rmse'],
        'all_cv_scores': {m: {'best_params': v['best_params'],
                              'cv_rmse': v['cv_rmse'],
                              'cv_rmse_std': v['cv_rmse_std'],
                              'cv_r2': v['cv_r2'],
                              'cv_r2_std': v['cv_r2_std'],
                              'cv_mae': v['cv_mae'],
                              'cv_mae_std': v['cv_mae_std'],
                              'train_time_s': v['train_time_s']}
                          for m, v in target_cv.items()},
    }

print(f'\nPhase A tuning complete.')
for t in TARGET_COLS:
    bm = phase_a_results[t]['best_model']
    rmse = phase_a_results[t]['cv_rmse']
    print(f'  {t}: {bm} (CV RMSE={rmse:.2f})')



  PHASE A — YS (9 N03 features, 345 samples)
  Selection criterion: lowest CV RMSE (scoring=neg_root_mean_squared_error)
  Model                               RMSE (CV)                     R²                    MAE     Time
  ──────────────────── ──────────────────────── ────────────────────── ────────────────────── ────────
  Dummy(mean)            129.29 ± 6.1   -0.0093


  0%|          | 0/50 [00:00<?, ?it/s]

  ExtraTrees              40.86 ± 3.9      0.8946 ± 0.0228      27.87 ± 2.8      53.4s  trials: 37 done/13 pruned
                       best params: {'n_estimators': 100, 'max_depth': 18, 'min_samples_split': 2, 'min_samples_leaf': 1}


  0%|          | 0/50 [00:00<?, ?it/s]

  GradientBoosting        43.44 ± 4.2      0.8817 ± 0.0211      30.67 ± 2.7     109.0s  trials: 47 done/3 pruned
                       best params: {'n_estimators': 800, 'max_depth': 8, 'learning_rate': 0.010777128814408872, 'subsample': 0.7127152635693225}


...

  LightGBM                 3.44 ± 0.5      0.7635 ± 0.0502       2.36 ± 0.4      60.2s  trials: 50 done/0 pruned
                       best params: {'n_estimators': 550, 'num_leaves': 65, 'min_child_samples': 17, 'learning_rate': 0.07030008609337791, 'subsample': 0.8462495016315642, 'colsample_bytree': 0.8733478521707003, 'reg_lambda': 9.88571225053242, 'reg_alpha': 0.001539052028195474}
  LassoCV                  5.58 ± 0.2      0.3859 ± 0.0983       4.06 ± 0.3       0.2s


  0%|          | 0/50 [00:00<?, ?it/s]

  RandomForest             3.94 ± 0.7      0.6969 ± 0.0474       2.57 ± 0.4      63.7s  trials: 48 done/2 pruned
                       best params: {'n_estimators': 300, 'max_depth': 17, 'min_samples_split': 2, 'min_samples_leaf': 1}


  0%|          | 0/50 [00:00<?, ?it/s]

  KNeighbors               4.24 ± 0.6      0.6443 ± 0.0496       2.84 ± 0.4       0.7s  trials: 34 done/16 pruned
                       best params: {'n_neighbors': 5, 'weights': 'distance'}
  >>> Best: LightGBM (CV RMSE=3.44)

Phase A tuning complete.
  YS: ExtraTrees (CV RMSE=40.86)
  UTS: ExtraTrees (CV RMSE=35.93)
  El: LightGBM (CV RMSE=3.44)


## 3. Phase A — Holdout Evaluation (single-split reference)

Train winner on full dev set (345 samples), evaluate on holdout (149 samples).
This is a **single-split reference** — Phase B provides robust 50-split stats.

In [4]:
print(f'\n{"="*80}')
print(f'  PHASE A — Holdout Evaluation (seed=42 single-split reference)')
print(f'  Train: {len(df_train)} samples | Test: {len(df_holdout)} samples')
print(f'{"="*80}')
print(f'  {"Target":8s} {"Model":20s} {"R²":>8s} {"RMSE":>8s} {"MAE":>8s}')
print(f'  {"─"*8} {"─"*20} {"─"*8} {"─"*8} {"─"*8}')

holdout_results = {}
for target_idx, target_name in enumerate(TARGET_COLS):
    winner = phase_a_results[target_name]['best_model']
    best_params = phase_a_results[target_name]['best_params']
    sel_feat_names = selected_sets[target_name]

    X_tr = build_X(sel_feat_names, df_train, X_dev)
    y_tr = y_dev[:, target_idx]
    X_te = build_X(sel_feat_names, df_holdout, X_holdout)
    y_te = y_holdout[:, target_idx]

    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_te_s = sc.transform(X_te)

    model = _build_tuned_model(winner, best_params)
    model.fit(X_tr_s, y_tr)
    yp = model.predict(X_te_s)

    r2 = r2_score(y_te, yp)
    rmse = np.sqrt(mean_squared_error(y_te, yp))
    mae = mean_absolute_error(y_te, yp)

    holdout_results[target_name] = {'r2': float(r2), 'rmse': float(rmse),
                                     'mae': float(mae)}
    print(f'  {target_name:8s} {winner:20s} {r2:8.4f} {rmse:8.2f} {mae:8.2f}')

print(f'  NOTE: Single-split reference. Phase B provides robust 50-split statistics.')
for t in TARGET_COLS:
    print(f'  {t}: {phase_a_results[t]["best_model"]} '
          f'(CV RMSE={phase_a_results[t]["cv_rmse"]:.2f}, '
          f'Holdout R²={holdout_results[t]["r2"]:.4f})')

print(f'\nPhase A complete — all design choices LOCKED for Phase B.')


  PHASE A — Holdout Evaluation (seed=42 single-split reference)
  Train: 345 samples | Test: 149 samples
  Target   Model                      R²     RMSE      MAE
  ──────── ──────────────────── ──────── ──────── ────────
  YS       ExtraTrees             0.8431    47.31    32.41
  UTS      ExtraTrees             0.8670    40.00    28.03
  El       LightGBM               0.7924     3.52     2.47
  NOTE: Single-split reference. Phase B provides robust 50-split statistics.
  YS: ExtraTrees (CV RMSE=40.86, Holdout R²=0.8431)
  UTS: ExtraTrees (CV RMSE=35.93, Holdout R²=0.8670)
  El: LightGBM (CV RMSE=3.44, Holdout R²=0.7924)

Phase A complete — all design choices LOCKED for Phase B.


## 4. Phase B — 50-Split Robust Validation ★ MAIN RESULT

50 independent random 70/30 splits on full cleaned data (494 samples, NaN
targets preserved). Each split runs independently — **imputation happens after
splitting** to prevent information leakage:

```
split(seed[i]) → MICE(ExtraTrees, fit on train) → StandardScaler(fit on train)
               → train(locked model + params) → evaluate(R², RMSE, MAE)
```

MICE stacks selected features X + target y. X has no NaN — MICE uses
fully-observed X to predict and fill missing y. Same pattern as N01 §6.

**Locked**: imputer type = ExtraTrees (N01), features = N03 SFFS,
model + hyperparams = Phase A. Each target evaluated independently.
Split seeds: `[42000, 42001, ..., 42049]`.

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

N_SPLITS = 50
TEST_SIZE = 0.30

# ── MICE imputer: read N01-selected estimator type ──
with open(os.path.join(DATA_DIR, 'best_imputer.json')) as f:
    imputer_info = json.load(f)
BEST_IMPUTER_NAME = imputer_info['best_imputer']  # ExtraTrees (locked by N01)
assert BEST_IMPUTER_NAME == 'ExtraTrees', f'Expected ExtraTrees, got {BEST_IMPUTER_NAME}'
_mice_est = ExtraTreesRegressor(n_estimators=100, random_state=RNG_SEED, n_jobs=-1)
print(f'MICE imputer: {BEST_IMPUTER_NAME} (from N01 best_imputer.json)')

# Pre-generate split seeds
split_seeds = [RNG_SEED * 1000 + i for i in range(N_SPLITS)]

# Storage: per target, per metric, list of 50 values
split_results = {t: {'r2': [], 'rmse': [], 'mae': []} for t in TARGET_COLS}
# Per-split predictions (y_true, y_pred) for scatter plots
phase_b_preds = {}

print(f'\n{"="*90}')
print(f'  PHASE B — 50-Split Robust Validation')
print(f'  {N_SPLITS} x random train_test_split (test_size={TEST_SIZE})')
print(f'  Per-split: split → MICE({BEST_IMPUTER_NAME}) → StandardScaler → train → evaluate')
print(f'  MICE fit on train only, transform test — no information leakage')
print(f'{"="*90}')

for target_idx, target_name in enumerate(TARGET_COLS):
    winner = phase_a_results[target_name]['best_model']
    best_params = phase_a_results[target_name]['best_params']
    sel_feat_names = selected_sets[target_name]

    # Build feature matrix from full raw dataset (NaN targets preserved)
    X_sel_full = build_X(sel_feat_names, df_full, X_full)
    y_t_full = y_full[:, target_idx]

    print(f'\n  ┌─ Target: {target_name}  |  Model: {winner}  |  {len(sel_feat_names)} features')
    print(f'  │  NaN in target: {np.isnan(y_t_full).sum()}/{len(y_t_full)}')

    r2_list, rmse_list, mae_list = [], [], []
    y_true_splits, y_pred_splits = [], []
    n_mice_applied = 0

    for split_i in range(N_SPLITS):
        # Step 1: Split raw data (NaN preserved — imputation happens after split)
        Xtr_raw, Xte_raw, ytr_raw, yte_raw = train_test_split(
            X_sel_full, y_t_full, test_size=TEST_SIZE,
            random_state=split_seeds[split_i])

        # Step 2: MICE imputation — fit on train, transform both.
        # X has no NaN; MICE stacks X + y, uses X to predict missing y.
        # Same pattern as N01 §6.
        has_nan = np.isnan(ytr_raw).any() or np.isnan(yte_raw).any()
        if has_nan:
            Xy_tr = np.column_stack([Xtr_raw, ytr_raw])  # stack X + y
            Xy_te = np.column_stack([Xte_raw, yte_raw])
            imp = IterativeImputer(estimator=_mice_est,
                                   random_state=split_seeds[split_i],
                                   max_iter=10, tol=1e-3)
            Xy_tr_i = imp.fit_transform(Xy_tr)   # fit on train (X → y)
            Xy_te_i = imp.transform(Xy_te)        # transform test only
            Xtr = Xy_tr_i[:, :-1]; ytr = Xy_tr_i[:, -1]
            Xte = Xy_te_i[:, :-1]; yte = Xy_te_i[:, -1]
            n_mice_applied += 1
        else:
            Xtr, ytr = Xtr_raw, ytr_raw
            Xte, yte = Xte_raw, yte_raw

        # Step 3: Scale (fit on train only)
        sc = StandardScaler()
        Xtr_s = sc.fit_transform(Xtr)
        Xte_s = sc.transform(Xte)

        # Step 4: Train & evaluate
        model = _build_tuned_model(winner, best_params)
        model.fit(Xtr_s, ytr)
        yp = model.predict(Xte_s)

        r2_list.append(float(r2_score(yte, yp)))
        rmse_list.append(float(np.sqrt(mean_squared_error(yte, yp))))
        mae_list.append(float(mean_absolute_error(yte, yp)))

        # Store per-split true vs predicted for scatter visualization
        y_true_splits.append(yte)
        y_pred_splits.append(yp)

        if (split_i + 1) % 10 == 0:
            print(f'  │  split {split_i+1:2d}/{N_SPLITS}: '
                  f'R²={np.mean(r2_list[-10:]):.4f} ± {np.std(r2_list[-10:]):.4f}, '
                  f'RMSE={np.mean(rmse_list[-10:]):.2f} ± {np.std(rmse_list[-10:]):.2f}')

    split_results[target_name]['r2'] = r2_list
    split_results[target_name]['rmse'] = rmse_list
    split_results[target_name]['mae'] = mae_list

    # Concatenate all test splits for scatter plot
    phase_b_preds[target_name] = {
        'y_true': np.concatenate(y_true_splits),
        'y_pred': np.concatenate(y_pred_splits),
    }

    print(f'  │  MICE applied in {n_mice_applied}/{N_SPLITS} splits')
    print(f'  ├─ R²   = {np.mean(r2_list):.4f} ± {np.std(r2_list):.4f}')
    print(f'  ├─ RMSE = {np.mean(rmse_list):.2f} ± {np.std(rmse_list):.2f}')
    print(f'  └─ MAE  = {np.mean(mae_list):.2f} ± {np.std(mae_list):.2f}')

# Save per-split predictions for standalone visualization (e.g. 09_visualization.ipynb)
np.savez(os.path.join(DATA_DIR, 'plot_phase_b.npz'),
         YS_true=phase_b_preds['YS']['y_true'], YS_pred=phase_b_preds['YS']['y_pred'],
         UTS_true=phase_b_preds['UTS']['y_true'], UTS_pred=phase_b_preds['UTS']['y_pred'],
         El_true=phase_b_preds['El']['y_true'], El_pred=phase_b_preds['El']['y_pred'])
print(f'  Saved per-split predictions to data/plot_phase_b.npz')

print(f'\nPhase B 50-split loop complete.')

MICE imputer: ExtraTrees (from N01 best_imputer.json)

  PHASE B — 50-Split Robust Validation
  50 x random train_test_split (test_size=0.3)
  Per-split: split → MICE(ExtraTrees) → StandardScaler → train → evaluate
  MICE fit on train only, transform test — no information leakage

  ┌─ Target: YS  |  Model: ExtraTrees  |  9 features
  │  NaN in target: 49/494
  │  split 10/50: R²=0.9033 ± 0.0187, RMSE=38.25 ± 3.80
  │  split 20/50: R²=0.9046 ± 0.0217, RMSE=37.20 ± 3.59
  │  split 30/50: R²=0.8989 ± 0.0221, RMSE=38.93 ± 3.40
  │  split 40/50: R²=0.9096 ± 0.0155, RMSE=37.46 ± 3.14
  │  split 50/50: R²=0.8997 ± 0.0151, RMSE=38.54 ± 2.54
  │  MICE applied in 50/50 splits
  ├─ R²   = 0.9032 ± 0.0192
  ├─ RMSE = 38.08 ± 3.39
  └─ MAE  = 25.56 ± 2.27

  ┌─ Target: UTS  |  Model: ExtraTrees  |  5 features
  │  NaN in target: 15/494
  │  split 10/50: R²=0.8907 ± 0.0224, RMSE=36.60 ± 2.85
  │  split 20/50: R²=0.8839 ± 0.0180, RMSE=36.80 ± 2.38
  │  split 30/50: R²=0.8944 ± 0.0164, RMSE=35.76 ± 2

In [6]:
# ══════════════════════════════════════════════════════════════
# Phase B -- Statistical Summary
# ══════════════════════════════════════════════════════════════

print(f'\n{"="*70}')
print(f'  FINAL RESULTS -- 50-Split Robust Validation')
print(f'  Mean ± Std')
print(f'  Each target evaluated independently with its own model')
print(f'{"="*70}')
print(f'  {"Target":8s} {"Model":20s} {"R²":>24s} {"RMSE":>24s}')
print(f'  {"─"*8} {"─"*20} {"─"*24} {"─"*24}')

for t in TARGET_COLS:
    r2s = split_results[t]['r2']
    rmses = split_results[t]['rmse']
    model_name = phase_a_results[t]['best_model']

    r2_str = f'{np.mean(r2s):.4f} ± {np.std(r2s):.4f}'
    rmse_str = f'{np.mean(rmses):.2f} ± {np.std(rmses):.2f}'

    print(f'  {t:8s} {model_name:20s} {r2_str:>24s} {rmse_str:>24s}')

print(f'{"="*70}')


  FINAL RESULTS -- 50-Split Robust Validation
  Mean ± Std
  Each target evaluated independently with its own model
  Target   Model                                      R²                     RMSE
  ──────── ──────────────────── ──────────────────────── ────────────────────────
  YS       ExtraTrees                    0.9032 ± 0.0192             38.08 ± 3.39
  UTS      ExtraTrees                    0.8914 ± 0.0195             36.37 ± 2.96
  El       LightGBM                      0.7479 ± 0.0533              3.71 ± 0.39


## 5. Summary & Next Steps

**Phase A** (seed=42, 345 samples): 6 models (top 6 from N01) tuned via Optuna
TPE per target on N03 SFFS features. Best model = lowest CV RMSE → locked.

**Phase B** ★ (50 splits on 494 raw samples): Per-split MICE → scale → train
→ evaluate with locked design choices. Imputation after splitting — no leakage.
Each target independent.

**Next**: `05_model_explanation.ipynb` — SHAP interpretation.

In [7]:
# ══════════════════════════════════════════════════════════════
# Save all artifacts
# ══════════════════════════════════════════════════════════════

# 1. Save best models (trained on full imputed dev set, for deployment)
for target_idx, target_name in enumerate(TARGET_COLS):
    winner = phase_a_results[target_name]['best_model']
    best_params = phase_a_results[target_name]['best_params']
    sel_feat_names = selected_sets[target_name]

    X_sel = build_X(sel_feat_names, df_train, X_dev)
    y_t = y_dev[:, target_idx]

    sc = StandardScaler()
    X_sel_s = sc.fit_transform(X_sel)

    model = _build_tuned_model(winner, best_params)
    model.fit(X_sel_s, y_t)

    pkg = {
        'model': model, 'scaler': sc,
        'features': sel_feat_names,
        'model_name': winner, 'best_params': best_params,
        'n_features': len(sel_feat_names),
        'repeated_split_r2_mean': float(np.mean(split_results[target_name]['r2'])),
        'repeated_split_r2_std': float(np.std(split_results[target_name]['r2'])),
        'repeated_split_rmse_mean': float(np.mean(split_results[target_name]['rmse'])),
        'rng_seed': RNG_SEED,
    }
    with open(os.path.join(MODEL_DIR, f'model_{target_name}.pkl'), 'wb') as f:
        pickle.dump(pkg, f)
    size_kb = os.path.getsize(os.path.join(MODEL_DIR, f'model_{target_name}.pkl')) / 1024
    print(f'Saved model_{target_name}.pkl ({winner}, {len(sel_feat_names)} features, {size_kb:.1f} KB)')

# 2. Save model selection results (Phase A)
msr = {}
for t in TARGET_COLS:
    msr[t] = {
        'best_model': phase_a_results[t]['best_model'],
        'best_params': phase_a_results[t]['best_params'],
        'cv_rmse': phase_a_results[t]['cv_rmse'],
        'holdout_r2': holdout_results[t]['r2'],
        'holdout_rmse': holdout_results[t]['rmse'],
        'holdout_mae': holdout_results[t]['mae'],
        'all_cv_scores': phase_a_results[t]['all_cv_scores'],
    }
msr['rng_seed'] = RNG_SEED
msr['n_dev_samples'] = len(df_train)
msr['n_holdout_samples'] = len(df_holdout)

with open(os.path.join(DATA_DIR, 'model_selection_results.json'), 'w') as f:
    json.dump(msr, f, indent=2)
print('Saved model_selection_results.json')

# 3. Save repeated split results (Phase B)
rsr = {}
for t in TARGET_COLS:
    rsr[t] = {
        'model': phase_a_results[t]['best_model'],
        'n_features': len(selected_sets[t]),
        'r2_mean': float(np.mean(split_results[t]['r2'])),
        'r2_std': float(np.std(split_results[t]['r2'])),
        'r2_all': [float(v) for v in split_results[t]['r2']],
        'rmse_mean': float(np.mean(split_results[t]['rmse'])),
        'rmse_std': float(np.std(split_results[t]['rmse'])),
        'rmse_all': [float(v) for v in split_results[t]['rmse']],
        'mae_mean': float(np.mean(split_results[t]['mae'])),
        'mae_std': float(np.std(split_results[t]['mae'])),
        'mae_all': [float(v) for v in split_results[t]['mae']],
        'best_params': phase_a_results[t]['best_params'],
    }
rsr['n_splits'] = N_SPLITS
rsr['test_size'] = TEST_SIZE
rsr['imputer'] = f'{BEST_IMPUTER_NAME} (N01)'
rsr['split_seeds'] = split_seeds
rsr['rng_seed'] = RNG_SEED
rsr['n_full_samples'] = len(df_full)

with open(os.path.join(DATA_DIR, 'repeated_split_results.json'), 'w') as f:
    json.dump(rsr, f, indent=2)
print('Saved repeated_split_results.json')

# Final summary
print(f'\n{"="*70}')
print('  N04 PIPELINE COMPLETE')
print(f'{"="*70}')
print(f'  Phase A (dev set): seed=42, {len(df_train)} samples (imputed)')
print(f'  Phase B (50-split): {N_SPLITS} x train_test_split, per-split MICE on {len(df_full)} raw samples')
print(f'  Imputer: ExtraTrees (N01)')
for t in TARGET_COLS:
    w = phase_a_results[t]['best_model']
    r2_m = np.mean(split_results[t]['r2'])
    r2_s = np.std(split_results[t]['r2'])
    rmse_m = np.mean(split_results[t]['rmse'])
    print(f'  {t}: {w} — R² = {r2_m:.4f} ± {r2_s:.4f}, '
          f'RMSE = {rmse_m:.2f} ({len(selected_sets[t])} features)')
print(f'{"="*70}')
print(f'\nNext: 05_model_explanation.ipynb')


Saved model_YS.pkl (ExtraTrees, 9 features, 4624.3 KB)
Saved model_UTS.pkl (ExtraTrees, 5 features, 17470.1 KB)
Saved model_El.pkl (LightGBM, 7 features, 785.3 KB)
Saved model_selection_results.json
Saved repeated_split_results.json

  N04 PIPELINE COMPLETE
  Phase A (dev set): seed=42, 345 samples (imputed)
  Phase B (50-split): 50 x train_test_split, per-split MICE on 494 raw samples
  Imputer: ExtraTrees (N01)
  YS: ExtraTrees — R² = 0.9032 ± 0.0192, RMSE = 38.08 (9 features)
  UTS: ExtraTrees — R² = 0.8914 ± 0.0195, RMSE = 36.37 (5 features)
  El: LightGBM — R² = 0.7479 ± 0.0533, RMSE = 3.71 (7 features)

Next: 05_model_explanation.ipynb
